<div style="border-left: 6px solid #00356B; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-bottom: 5px; color: #00356B"><strong>Assignment 3:</strong> Part 1 (Single-Model Calibration)</h1>
  <span style="font-size: 1.2em; color: #444; font-weight: bold">S&DS 5350 | Social Algorithms</span>
  <br><br>
  <strong>Primary:</strong> Brandon Tran (bat53)
  <br>
  <strong>Partner:</strong> Cailey Bobadilla (cjb239)
</div>

---

*Mood for this part:*

<iframe data-testid="embed-iframe" style="border-radius:12px" src="https://open.spotify.com/embed/track/3aHFzz6nnlhhgk2dX1pf2g?utm_source=generator" width="40%" height="152" frameBorder="0" allowfullscreen="" allow="autoplay; clipboard-write; encrypted-media; fullscreen; picture-in-picture" loading="lazy"></iframe>

#### I.1 | Warm-Up: One Simple Scattergories Question
We will use a single local Ollama player model, `quen2.5:7b`, to:

1. Produce a name of the week, with the goal of making the distribution of outcomes as close to uniform as possible.
2. Sample 500 generations per temperature over a grid of temperatures.
3. Plot histograms of answer frequencies at each temperature.

Recall that temperature, $T$, in this context is a hyperparameter that controls the randomness (or "creativity") of the model's predicitons by scaling the probabilities of the next potential tokens. $T=1$ instructs the model to sample from a natural, unscaled probability distribution. As $T \to 0$, the model becomes more deterministic, with the probability of the highest-scoring token appraoching $1$. Conversely, as $T \to \infty$, the differences between token scores shrink; the probability distribution becomes uniform such that every token has an equal chance of being selected.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

from reference.assignment3_starter import(
    ollama_generate,
    normalize_answer,
    entropy_from_counts,
    kl_to_uniform
)

%matplotlib inline

MODEL = 'qwen2.5:7b'
SAMPLES = 500
TEMPERATURES = [0.1, 0.5, 1.0, 1.5, 2.0]
VALID_DAYS = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday,',
              'saturday', 'sunday']

In [ ]:
prompt = (
    "Return a day of the week in lowercase. "
    "Output only the weekday (no additional text). "
    "For example, you would output: saturday"
)

results = []

for temp in TEMPERATURES:
    print(f"Running Temp: {temp}")
    counts = Counter()

    for i in range(SAMPLES):
        raw_ans = ollama_generate(
            model = MODEL,
            prompt = prompt,
            temperature = temp,
            top_k = 40,           # Only consider the top 40 most likely words.
            max_tokens = 8        # Enough for one word (approx. 32 characters)
        )

        ans = normalize_answer(raw_ans)
        ans = ans.split()[0] if ans else "invalid"   # In case we get >1 word.

        counts[ans] += 1
        results.append({"temperature": temp, "raw": raw_ans, "answer": ans})

        if (i+1) % 100 == 0:
            print(f"Completed {i+1}/{SAMPLES} samples.")
        
    print(f"Unique answers at temp {temp}: {len(counts)}")

df = pd.DataFrame(results)


Running Temp: 0.1
